# Datamine DTMCUT Process Example

This notebook demonstrates how to configure and run the **`dtmcut`** process wrapper in `dmstudio`.

## Process Description

## DTMCUT

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This process allows the modelling and evaluation of cut and fill volumes, based on original and updated wireframe surfaces (DTMs).

Both cut and fill volumes are separately evaluated, and both are described in an output block model, whose block sizes are controlled by an input block model prototype. This input prototype can also define a rotated model structure, which will be honoured as supplied.

A supplied cut attribute field is used for the assignment of the cut and fill volumes. The user therefore supplies the name of this new (numeric) field, and what values will be assigned within 'cut' blocks and 'filled' blocks. This enables clear identification of cut and fill volumes, for both evaluation and plotting/display purposes.

An optional perimeter file may also be supplied, to split the cut and fill volumes into separate regions for evaluation purposes. This allows the cuts to be evaluated in separate mining strips or partitions. These perimeters are treated as boundaries as viewed in an XY plan i.e. the elevations of the perimeters are ignored. If such perimeters are supplied, an optional attribute field can also be defined, to further divide the evaluation data as required.

The degree of accuracy is controlled by the supplied cell splitting parameter. This parameter is utilized in exactly the same manner as the SPLITS parameter in the [TRIFIL](<trifil.md>) process.

Density values must be supplied for the cut and fill material, for use in the subsequent tonnage calculation. These values will also be placed into the density field (whose name is user-defined) of the output cut/fill block model.

The output results file contains all the evaluated tonnages and volumes, split by cut, fill and perimeter attribute. This results file is automatically also written out as a .csv file, for easy import into spreadsheet programs.

### Input Files:

* **wiretr1** (Wireframe triangle):
  Triangle file of original wireframe surface (DTM).
  Required=Yes

* **wirept1** (Wireframe points):
  Point file of original wireframe surface (DTM).
  Required=Yes

* **wiretr2** (Wireframe triangle):
  Triangle file of update wireframe surface (DTM).
  Required=Yes

* **wirept2** (Wireframe points):
  Point file of update wireframe surface (DTM).
  Required=Yes

* **proto** (Block model):
  Input block model prototype.
  Required=Yes

* **perimin** (Perimeter file):
  Optional input perimeter file controlling sub-division of cut-and-fill volumes.
  Required=No

### Output Files:

* **cutmodou** (Block model):
  Output block model of cut and fill volumes.
  Required=Yes

* **results** (Results file):
  Output evaluation results data file.
  Required=Yes

### Fields:

* **density** (Numeric : MODELIN):
  Density field in output block model.
  Default=Undefined
  Required=Yes

* **cutfld** (Numeric :):
  Output numeric field defining cut and fill volumes.
  Default=Undefined
  Required=Yes

* **attrib** (Any : PERIMIN):
  Optional attribute field from input perimeter file.
  Default=Undefined
  Required=No

### Parameters:

* **cutden**:
  Density of cut volumes.
  Range=0,99999
  Values=
  Default=1
  Required=Yes

* **fillden**:
  Density of filled volumes.
  Range=0,99999
  Values=
  Default=1
  Required=Yes

* **splits**:
  Subcell splitting of cut and fill block model.
  Range=0,3
  Values=
  Default=0
  Required=Yes

* **cutval**:
  Value assigned to **CUTFLD** for cells inside cut volume.
  Range=
  Values=
  Default=-1
  Required=Yes

* **fillval**:
  Value assigned to **CUTFLD** for cells inside fill volume.
  Range=
  Values=
  Default=1
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('dtmcut')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute dtmcut
print("Running dtmcut...")
dm_cmd.dtmcut(
    wiretrs_i=['t_SurfaceTriangles'],  # required
    wirepts_i=['t_SurfacePointsPt'],  # required
    proto_i='t_mod1',  # required
    perimin_i='optional',  # required
    cutmodou_o='t_dtmcut_out',  # required
    results_o=['t_dtmcut_out'],  # required
    density_f='optional',  # required
    cutfld_f='optional',  # required
    attrib_f='optional',  # required
    cutden_p='required_val',  # required
    fillden_p='required_val',  # required
    splits_p='required_val',  # required
    cutval_p='required_val',  # required
    fillval_p='required_val',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("dtmcut execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_dtmcut_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")